# L09 · NB 04 — Build a semantic search engine

> *We have embeddings. Now let's wrap them in a clean, production-style search engine and benchmark it head-to-head against TF-IDF — the classical baseline that every real search system still uses for exact-match queries.*

⏱️ **~40 min** in-class code-along. The 🟡 Extension section is optional self-study.

### 👟 Step into Sarah's shoes

**Where we left off (NB 03).** Sarah proved a pretrained model beats keyword search on her benchmark. The brain works. But a pile of embeddings in a notebook is **not a search engine** — and Marcus can't ship a science experiment.

> *"Great, it's smart. Now: how do I package this so the website can call it? Do I throw away the old keyword approach entirely? And what does a real production search system actually look like?"*

**What Sarah wants to achieve here.** Turn the loose pieces into a clean, reusable **search engine**, stress-test it honestly against the classical baseline (**TF-IDF**), and design the real-world wiring NorthStar would actually deploy.

**How she'll do it — step by step:**

1. **Wrap it in a class** — `SemanticSearch` with a tidy API: `.add()`, `.search()`, `.search_with_filters()`. This is the shippable artifact.
2. **Run real queries** — including ones with *no* category keyword ("something cosy to wear hiking") to show it generalises.
3. **Add structured filters** — restrict to a category/price first, *then* rank. Filters + semantics are complementary, not rivals.
4. **Build TF-IDF** — the classical "smart Ctrl-F" baseline every real engine still uses.
5. **Benchmark head-to-head** — semantic vs TF-IDF on the same 8 queries. See exactly which queries each wins.
6. **Find where each wins** — synonyms favour semantics; exact product names favour TF-IDF.
7. **Draw the production diagram** — the **hybrid** design: run both, union the candidates, filter, re-rank, return.

> The lesson that lands: production search isn't "semantic OR keyword" — it's **both, plus filters, plus a re-ranker**. Sarah finishes with a working engine *and* the architecture to scale it.

## 1 · Setup — load model, catalogue, cached embeddings

In [1]:
# --- Colab setup: install sentence-transformers if missing (no-op in the local dsai-m3 env) ---
try:
    import sentence_transformers  # library fetches models from the Hugging Face Hub
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'sentence-transformers'], check=True)

import os
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'

import json
import time
import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# (Removed the old one-CPU-thread cap — it made CPU-only machines much slower.)

_LOCAL = 'data/northstar_catalogue.csv'
_URL = 'https://raw.githubusercontent.com/su-ntu-ctp/6m-data-3.9-Natural-Language-Processing/main/notebooks/data/northstar_catalogue.csv'
df = pd.read_csv(_LOCAL if os.path.exists(_LOCAL) else _URL)

# Reload the same model we used in NB 03.
# Offline-first: if it's already in the local Hugging Face cache, load it from disk
# (no network); only hit the Hub to download when it isn't cached yet.
_MODEL_NAME = 'all-MiniLM-L6-v2'
try:
    model = SentenceTransformer(_MODEL_NAME, local_files_only=True)
    print(f"Model '{_MODEL_NAME}' already cached — loaded from disk, skipped download.")
except Exception:
    print(f"Model '{_MODEL_NAME}' not in cache — downloading from the Hugging Face Hub…")
    model = SentenceTransformer(_MODEL_NAME)
    print("Download complete and cached for next time.")

# Load or recompute catalogue embeddings (cached from NB 03).
# Recompute if the file is missing OR empty/corrupt, so a stale 0-byte
# file can never crash this cell.
def _have_valid(path):
    return os.path.exists(path) and os.path.getsize(path) > 0

catalogue_emb = None
if _have_valid('catalogue_embeddings.npy'):
    try:
        catalogue_emb = np.load('catalogue_embeddings.npy')  # reuse the cached file, no recompute
        print(f"Loaded cached embeddings: {catalogue_emb.shape}")
    except Exception as e:
        print(f"Cached embeddings unreadable ({e}); recomputing.")

if catalogue_emb is None:
    # No usable cache — embed the corpus from scratch and save it for next time
    docs = (df['name'] + ' — ' + df['description']).tolist()
    catalogue_emb = model.encode(docs)
    np.save('catalogue_embeddings.npy', catalogue_emb)
    print(f"Computed embeddings: {catalogue_emb.shape}")

# The 8-query benchmark from NB 01, inlined so this notebook is self-contained
# (run it standalone in Colab without needing NB 01's output file).
ground_truth = {
    "blue summer dress":         "Lila Floral Sundress",
    "warm winter jumper":        "Roan Cable Jumper",
    "smart office shoes":        "Loam Ankle Boots",
    "lightweight rain jacket":   "Trail Windbreaker",
    "cosy throw for sofa":       "Wool Throw Blanket",
    "gym leggings":              "Storm Yoga Leggings",
    "beach holiday outfit":      "Cassia Maxi Gown",
    "running trainers":          "Cloud Running Trainers",
}

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Model 'all-MiniLM-L6-v2' already cached — loaded from disk, skipped download.
Loaded cached embeddings: (76, 384)


## 2 · A clean search-engine class

One class, three methods. Everything that follows uses this object.

### 🐍 New to Python? What's a "class" and a "method"?

Think of a **class** as a *blueprint* — a template that bundles together some **data** and the **actions** you can perform on that data. A **method** is just a function that lives *inside* a class and operates on that bundled data.

A common analogy is a car factory:

- The **class** (`Car`) is the blueprint — it describes what every car has (wheels, an engine) and what it can do (drive, brake).
- An **object** (or *instance*) is one actual car built from that blueprint. You can build many cars from one blueprint.
- A **method** (`car.drive()`) is an action the car can perform.

In our code:

| Concept | In this notebook | Plain-English meaning |
|---|---|---|
| **Class** | `SemanticSearch` | The blueprint for "a search engine" |
| **Object / instance** | `engine = SemanticSearch(model)` | One actual search engine we built and can use |
| **Attribute** (the data) | `self.df`, `self.embeddings` | What the engine *remembers* — the catalogue and its vectors |
| **Method** (the actions) | `.add()`, `.search()`, `.search_with_filters()` | What the engine can *do* |

A few keywords you'll see below:

- **`class SemanticSearch:`** — declares the blueprint.
- **`def __init__(self, ...)`** — the *constructor*: special method that runs once when you build an object (`SemanticSearch(model)`). It sets up the starting data.
- **`self`** — refers to *this particular object*. `self.df` means "this engine's own catalogue." Every method takes `self` as its first argument so it can read and update the object's own data.
- **`engine.search("blue dress")`** — *calling a method* on an object: ask `engine` to run its `search` action.

**Why bundle it into a class instead of loose functions?** Because the data (catalogue + embeddings) and the actions (search) belong together. Once you've built the `engine` object, you just call `engine.search(...)` — you never have to pass the embeddings around by hand. That's exactly how real, shippable search code is organised.

In [2]:
# SemanticSearch bundles the three methods you'd structure real search code around
class SemanticSearch:
    """A minimal semantic search engine over a product catalogue.

    add(df, text_cols)       — embed the corpus once and store
    search(query, top=10)    — top-K by cosine similarity
    search_with_filters(...) — top-K within structured filters
    """
    def __init__(self, model):
        self.model = model        # the sentence-transformer that turns text → vectors
        self.df = None            # will hold the product table (filled in by .add())
        self.embeddings = None    # will hold one 384-dim vector per product (filled in by .add())

    def add(self, df, text_cols=('name', 'description')):
        # Store a clean copy of the catalogue; reset_index so row 0,1,2… line up with embedding 0,1,2…
        self.df = df.reset_index(drop=True).copy()
        # Build the text to embed for each product by joining the chosen columns with ' — '
        # e.g. "Lila Floral Sundress — Lightweight floral frock perfect for warm summer days."
        docs = self.df[list(text_cols)].agg(' — '.join, axis=1).tolist()
        # Encode all product texts in one batch → a (n_products × 384) matrix. Done ONCE, up front.
        self.embeddings = self.model.encode(docs, show_progress_bar=False)
        return self               # return self so you can chain: SemanticSearch(model).add(df)

    def search(self, query, top=5):
        q = self.model.encode([query])                   # embed the query → shape (1, 384)
        sims = cosine_similarity(q, self.embeddings)[0]  # similarity of query vs every product → (n,)
        order = np.argsort(-sims)[:top]                  # indices sorted high→low score; keep top-K
        # Pull those rows from the table, attach their scores, and renumber 0..top-1 for display
        return self.df.iloc[order].assign(score=sims[order]).reset_index(drop=True)

    def search_with_filters(self, query, top=5, **filters):
        """Filter the corpus first, then rank by similarity."""
        q = self.model.encode([query])                   # embed the query (same as above)
        mask = pd.Series(True, index=self.df.index)      # start with every product allowed (all True)
        for col, val in filters.items():                 # apply each filter, e.g. category='dress'
            if isinstance(val, (list, tuple)):           # a list of allowed values → keep rows in it
                mask &= self.df[col].isin(val)
            else:                                        # a single value → keep rows that match exactly
                mask &= (self.df[col] == val)
        sub_idx = self.df.index[mask]                    # the row numbers that survived all filters
        if len(sub_idx) == 0:                            # nothing matched the filters →
            return self.df.iloc[:0]                      #   return an empty table (no results)
        sub_emb = self.embeddings[mask.values]           # keep only the surviving products' vectors
        sims = cosine_similarity(q, sub_emb)[0]          # rank the query WITHIN that subset only
        order = np.argsort(-sims)[:top]                  # top-K positions inside the subset
        # Map subset positions back to original rows, attach scores, renumber for display
        return self.df.iloc[sub_idx[order]].assign(score=sims[order]).reset_index(drop=True)

# Wire it up — re-uses our cached embeddings, just for cleanliness
engine = SemanticSearch(model)
# Skip re-encoding (we already have catalogue_emb from NB 03), so set the fields directly
engine.df = df.reset_index(drop=True).copy()   # same catalogue table the embeddings were built from
engine.embeddings = catalogue_emb              # reuse the cached (76 × 384) embedding matrix
print('Engine ready.')

Engine ready.


## 3 · Try a few queries

In [3]:
# Note 'cosy to wear hiking' and 'business attire for women': neither uses a product-category
# word, yet both return sensible products — semantic search delivering on its promise.
for q in ['blue summer dress', 'something cosy to wear hiking', 'business attire for women']:
    print(f"\nQuery: {q!r}")
    print(engine.search(q, top=5)[['category','name','score']].to_string(index=False))


Query: 'blue summer dress'
category                    name    score
   shirt       Frost Linen Shirt 0.547784
   dress    Marigold Shift Dress 0.540338
   dress    Lila Floral Sundress 0.467017
   dress Holly Knit Jumper Dress 0.466092
   shirt       Indigo Polo Shirt 0.460361

Query: 'something cosy to wear hiking'
category                 name    score
footwear Boulder Hiking Boots 0.500391
footwear     Aspen Snow Boots 0.482619
   shirt   Pine Flannel Shirt 0.464685
   shirt    Frost Linen Shirt 0.457966
footwear     Loam Ankle Boots 0.437125

Query: 'business attire for women'
category                 name    score
   dress     Cassia Maxi Gown 0.460027
   dress Sienna Bodycon Dress 0.458183
    knit Heather Crew Sweater 0.441692
   shirt Rose Puff Sleeve Top 0.432061
   shirt     Onyx Silk Blouse 0.417356


Note the second and third queries — *something cosy to wear hiking* and *business attire for women*. Neither uses any product-category keywords. Both pull plausible results. That's semantic search delivering on its promise.

## 4 · Filters on top of semantic ranking

In production you usually want to RESTRICT the corpus first (e.g., to one category, or one price range) and then rank within that subset. The `search_with_filters` method does this.

In [4]:
# Restricting to category='dress' drops a non-dress distractor (a linen shirt) and lifts Lila
# from rank 3 to rank 2 — structured filtering and semantic ranking are complementary.
print("Query: 'blue summer dress'   |   filter: category=='dress'")
# "blue summer dress" but only within the dress category
print(engine.search_with_filters('blue summer dress', top=5, category='dress')[['name','score']].to_string(index=False))

print()
print("Query: 'lightweight rain jacket'   |   filter: category in ['coat','activewear']")
print(engine.search_with_filters('lightweight rain jacket', top=5, category=['coat','activewear'])[['category','name','score']].to_string(index=False))

Query: 'blue summer dress'   |   filter: category=='dress'
                   name    score
   Marigold Shift Dress 0.540338
   Lila Floral Sundress 0.467017
Holly Knit Jumper Dress 0.466092
   Sienna Bodycon Dress 0.447115
       Cassia Maxi Gown 0.430717

Query: 'lightweight rain jacket'   |   filter: category in ['coat','activewear']
category                  name    score
    coat   Cinder Biker Jacket 0.561851
    coat   Verge Bomber Jacket 0.559493
    coat  Ember Quilted Jacket 0.531884
    coat     Tarn Waxed Jacket 0.498808
    coat Glacier Puffer Jacket 0.453990


With the category filter applied, the non-dress distractor (Frost Linen Shirt) is gone, so *Lila Floral Sundress* climbs from rank 3 to **rank 2**. It is still behind *Marigold Shift Dress* (a yellow dress) — the model over-weights the word "dress" relative to the colour "blue". So the filter helps but does **not** fully fix the colour-blindness; closing that gap needs a colour filter or a re-ranker on top.

**Production lesson:** structured filters and semantic ranking are complementary. Don't pick one. Use filters for what filters do well (exact field matching), use embeddings for the rest.

## 5 · TF-IDF — the classical baseline

Let's build a TF-IDF search engine over the same corpus and compare head-to-head. TF-IDF treats every word as independent but weights them by how rare they are across the corpus.

### 📖 What is TF-IDF, in plain English?

**TF-IDF** stands for **Term Frequency × Inverse Document Frequency**. It's a way to score how *important* a word is to one product, by combining two common-sense ideas. Think of it as a **much smarter Ctrl-F**: it still matches literal words (no understanding of meaning), but it's clever about *which* words matter.

**1 · Term Frequency (TF) — "how often does this word appear in this product?"**

If the word *"linen"* appears several times in a product's description, that product is probably *about* linen. More mentions → higher weight. Simple count.

**2 · Inverse Document Frequency (IDF) — "how rare is this word across the whole catalogue?"**

This is the clever half. A word that appears in *almost every* product (like *"the"*, *"soft"*, *"perfect"*) tells you almost nothing — so IDF gives it a **low** weight. A word that appears in only a *handful* of products (like *"espadrille"* or *"cobalt"*) is highly distinctive — so IDF gives it a **high** weight. "Inverse" = the rarer the word, the bigger its score.

**Multiply them together** and you get the punchline:

> A word scores high for a product only if it appears **often in that product** (high TF) **but rarely across the catalogue** (high IDF). Common filler words are automatically squashed; rare, meaningful words dominate the match.

That's the whole idea. It's why our `TfidfVectorizer` below uses `stop_words='english'` (drop ultra-common words entirely) and `ngram_range=(1, 2)` (also score two-word phrases like *"rain jacket"*, not just single words).

**The catch — and why it's only a baseline.** TF-IDF still treats every word as an opaque string. To it, *frock* and *dress* are as unrelated as *frock* and *spaceship* — it weights words, but has **zero notion of meaning** (exactly the NB 01 / NB 02 limitation). So it shines on **exact-token** queries (a precise product name, a rare keyword) and breaks on **synonym/paraphrase** queries — the mirror image of where the semantic model wins. That complementarity is the whole reason we benchmark them head-to-head and end up combining both.

In [5]:
# The classic TF-IDF baseline — think of it as a very smart Ctrl-F: it weights words but still
# treats them as independent strings with no meaning.
class TfidfSearch:
    def __init__(self):
        self.df = None
        # weight each word by how rare it is across the catalogue (rare words = strong signal)
        self.vec = TfidfVectorizer(stop_words='english', ngram_range=(1, 2))
        self.matrix = None

    def add(self, df, text_cols=('name', 'description')):
        self.df = df.reset_index(drop=True).copy()
        docs = self.df[list(text_cols)].agg(' '.join, axis=1).tolist()
        self.matrix = self.vec.fit_transform(docs)
        return self

    def search(self, query, top=5):
        q = self.vec.transform([query])
        sims = cosine_similarity(q, self.matrix)[0]
        order = np.argsort(-sims)[:top]
        return self.df.iloc[order].assign(score=sims[order]).reset_index(drop=True)

tfidf = TfidfSearch().add(df)
print(f"TF-IDF vocabulary size: {len(tfidf.vec.vocabulary_)}")
print(f"TF-IDF matrix shape   : {tfidf.matrix.shape}")

TF-IDF vocabulary size: 1271
TF-IDF matrix shape   : (76, 1271)


### TF-IDF on the same 'blue summer dress' query

In [6]:
# TF-IDF's bigrams and rarity-weighting beat the NB 01 keyword search, but it still can't find
# the Lila frock — its description never contains 'summer', 'dress' or 'blue'.
print("Query: 'blue summer dress'")
print()
print(tfidf.search('blue summer dress', top=5)[['category','name','score']].to_string(index=False))

Query: 'blue summer dress'

category                    name    score
   dress Holly Knit Jumper Dress 0.147678
   dress       Marina Wrap Dress 0.144762
   shirt       Frost Linen Shirt 0.124843
    knit    Cobalt V-Neck Jumper 0.122155
footwear Sand Espadrille Sandals 0.118865


TF-IDF's bigrams and IDF weighting are an improvement over the NB 01 keyword search, but it still can't find Lila Floral Sundress because the description never uses the words *summer*, *dress*, or *blue*.

## 6 · Head-to-head benchmark

Run all 8 ground-truth queries through both engines.

In [7]:
def evaluate(engine, ground_truth, top_k=5):
    """Return per-query top-K results and metric: top-1 + top-K hit rate."""
    results = []
    for q, expected in ground_truth.items():
        top = engine.search(q, top=top_k)
        top1_ok = top.iloc[0]['name'] == expected     # right answer at rank 1?
        topk_ok = expected in top['name'].tolist()    # right answer anywhere in top-K?
        results.append((q, expected, top.iloc[0]['name'], top1_ok, topk_ok))
    return results

# Run both engines over all 8 ground-truth queries
sem_results = evaluate(engine, ground_truth, top_k=5)
tf_results  = evaluate(tfidf,  ground_truth, top_k=5)

print(f"{'Query':30s} {'Semantic top-1':35s} {'TF-IDF top-1':35s} {'sem@1':>6s} {'tf@1':>6s}")
print('-' * 120)
for (q, exp, sem_top, sem_ok, sem_topk), (_, _, tf_top, tf_ok, tf_topk) in zip(sem_results, tf_results):
    print(f"  {q:28s} {sem_top:35s} {tf_top:35s} {'✅' if sem_ok else '❌':>6s} {'✅' if tf_ok else '❌':>6s}")

sem_top1 = sum(r[3] for r in sem_results)
tf_top1  = sum(r[3] for r in tf_results)
sem_top5 = sum(r[4] for r in sem_results)
tf_top5  = sum(r[4] for r in tf_results)

# Result: Semantic wins top-1 — its extra wins are the synonym-heavy queries where TF-IDF derails
print(f"\n{'Approach':12s} {'Top-1':>6s} {'Top-5':>6s}")
print(f"{'Semantic':12s} {sem_top1}/{len(sem_results):d}    {sem_top5}/{len(sem_results):d}")
print(f"{'TF-IDF':12s} {tf_top1}/{len(tf_results):d}    {tf_top5}/{len(tf_results):d}")

Query                          Semantic top-1                      TF-IDF top-1                         sem@1   tf@1
------------------------------------------------------------------------------------------------------------------------
  blue summer dress            Frost Linen Shirt                   Holly Knit Jumper Dress                  ❌      ❌
  warm winter jumper           Roan Cable Jumper                   Holly Knit Jumper Dress                  ✅      ❌
  smart office shoes           Loam Ankle Boots                    Cloud Running Trainers                   ✅      ❌
  lightweight rain jacket      Cinder Biker Jacket                 Ember Quilted Jacket                     ❌      ❌
  cosy throw for sofa          Wool Throw Blanket                  Wool Throw Blanket                       ✅      ✅
  gym leggings                 Storm Yoga Leggings                 Storm Yoga Leggings                      ✅      ✅
  beach holiday outfit         Cassia Maxi Gown             

**Headline result: Semantic around 6/8 top-1 and 7/8 top-5. TF-IDF around 4/8 on both.** (Your exact counts may differ by a query or so if your model version drifts — the gap is the point.)

Semantic wins decisively. The four common wins are the "easy" queries (cosy throw / gym leggings / beach holiday outfit / running trainers — where the right answer's name contains a content word from the query). The two semantic-only wins (warm winter jumper, smart office shoes) are exactly the synonym-heavy cases where TF-IDF goes off the rails — *jumper* matches "Holly Knit Jumper Dress" (a dress!) because TF-IDF doesn't know one is a noun-modifier and one is the head noun.

The two queries where both fail (*blue summer dress*, *lightweight rain jacket*) need the structured filters we explored in §4. We'll close that gap shortly.

## 7 · When each approach wins

The benchmark told us *who* wins overall. This section shows *why*, with three query types that map cleanly onto the strengths and blind spots of each engine — first two contrasting queries, then the sharpest case of all.

In [8]:
# Two contrasting queries, each one an engine's home turf (or hard case)
contrastive_queries = {
    "P0001": "soft cotton flowy garment with petal patterns",  # pure-synonym phrasing — trips up BOTH
    "P0002": "Marina Wrap Dress",                              # exact product name — TF-IDF's specialty
}

for pid, q in contrastive_queries.items():
    expected = df.loc[df['product_id'] == pid, 'name'].iloc[0]
    sem_top1 = engine.search(q, top=1).iloc[0]['name']
    tf_top1  = tfidf.search(q, top=1).iloc[0]['name']
    # find where the expected product actually lands in each full ranking
    sem_rank = engine.search(q, top=76).reset_index(drop=True)
    sem_rank = sem_rank.index[sem_rank['name'] == expected].tolist()
    sem_rank = sem_rank[0] + 1 if sem_rank else 'NOT IN TOP 76'
    tf_rank  = tfidf.search(q,  top=76).reset_index(drop=True)
    tf_rank  = tf_rank.index[tf_rank['name'] == expected].tolist()
    tf_rank  = tf_rank[0] + 1 if tf_rank else 'NOT IN TOP 76'

    print(f"Query: {q!r}")
    print(f"  Expected         : {expected}")
    print(f"  Semantic top-1   : {sem_top1}  (rank of expected: {sem_rank})")
    print(f"  TF-IDF top-1     : {tf_top1}  (rank of expected: {tf_rank})")
    print()

Query: 'soft cotton flowy garment with petal patterns'
  Expected         : Lila Floral Sundress
  Semantic top-1   : Indigo Polo Shirt  (rank of expected: 19)
  TF-IDF top-1     : Velvet Cushion Cover  (rank of expected: 17)

Query: 'Marina Wrap Dress'
  Expected         : Marina Wrap Dress
  Semantic top-1   : Marina Wrap Dress  (rank of expected: 1)
  TF-IDF top-1     : Marina Wrap Dress  (rank of expected: 1)



**Two cases so far:**

- **Synonym query** ("soft cotton flowy garment with petal patterns") → *both fail*. Semantic ranks Lila 19th, TF-IDF 17th. The query is metaphorical (*petal patterns* for floral, *flowy garment* for dress) — too far from the literal description for either to connect. Real customers rarely phrase searches this way, but it shows even semantics has limits, and motivates the filters + re-ranker layer.
- **Exact product name** ("Marina Wrap Dress") → *both win at rank 1*. A tie: TF-IDF matches the literal tokens, and semantic does too because identical words are also the nearest vectors.

So far no clear TF-IDF win. The next query delivers one.

### 🎯 Where TF-IDF decisively wins: exact product codes (SKUs)

Customers, warehouse staff, and support agents constantly paste an **exact identifier** — a SKU, a product code, an order number. Try searching for `P0047` (the code for *Sand Espadrille Sandals*).

A product code carries **no meaning** — it's an arbitrary string, so the semantic model has nothing to "understand" and returns near-random neighbours. TF-IDF treats `p0047` as one rare token that appears in exactly **one** product → instant, exact, top-1 match. This is the exact-token precision embeddings lack.

In [9]:
# To search by product code, the code must be part of the indexed text — so here we re-index BOTH
# engines on (product_id + name + description). Everything else is identical to before.
sku_cols = ('product_id', 'name', 'description')
sem_sku = SemanticSearch(model).add(df, text_cols=sku_cols)   # semantic engine, now aware of the codes
tf_sku  = TfidfSearch().add(df, text_cols=sku_cols)           # TF-IDF engine, same corpus

query = 'P0047'
expected = df.loc[df['product_id'] == query, 'name'].iloc[0]
print(f"Query: {query!r}   (the correct product is: {expected})\n")

# Where does the right product land in each ranking?
sem_full = sem_sku.search(query, top=len(df)).reset_index(drop=True)
sem_rank = sem_full.index[sem_full['product_id'] == query].tolist()
sem_rank = sem_rank[0] + 1 if sem_rank else 'NOT FOUND'

print("Semantic top-3 (no idea what a code 'means' → near-random):")
print(sem_sku.search(query, top=3)[['product_id', 'name', 'score']].to_string(index=False))
print(f"  → correct product's actual rank: {sem_rank}\n")

print("TF-IDF top-3 (exact-token match → nails it at rank 1):")
print(tf_sku.search(query, top=3)[['product_id', 'name', 'score']].to_string(index=False))

Query: 'P0047'   (the correct product is: Sand Espadrille Sandals)

Semantic top-3 (no idea what a code 'means' → near-random):
product_id                 name    score
     P0037 Cobalt V-Neck Jumper 0.445361
     P0045 Whisper Ballet Flats 0.429097
     P0006 Bea Tea-Length Frock 0.427048
  → correct product's actual rank: 76

TF-IDF top-3 (exact-token match → nails it at rank 1):
product_id                    name    score
     P0047 Sand Espadrille Sandals 0.202723
     P0001    Lila Floral Sundress 0.000000
     P0055     Driftwood Straw Hat 0.000000


**The three cases as one spectrum:**

| Query type | Semantic | TF-IDF | Why |
|---|---|---|---|
| Poetic synonym (*"flowy garment, petal patterns"*) | ✗ (rank 19) | ✗ (rank 17) | too metaphorical for either → needs filters / re-ranker |
| Exact name (*"Marina Wrap Dress"*) | ✓ (rank 1) | ✓ (rank 1) | a tie — literal words are also nearest vectors |
| **Product code (*"P0047"*)** | **✗ (rank 76)** | **✓ (rank 1)** | TF-IDF's home turf — a meaningless token only it can match exactly |

Semantic isn't bad at *exact* matches (it ties on the product name) — it's bad at *meaningless* tokens. That's the gap TF-IDF fills. Neither engine wins everywhere, which is exactly why production search runs **both**:

```
final_results = semantic_top_K  ∪  tfidf_top_K   # union of candidates
              → apply structured filters (category / colour / price)
              → re-rank by weighted blend + popularity + recency
              → return top-K
```

Hybrid retrieval: embeddings for semantic recall, TF-IDF for exact-token precision, filters for hard constraints, a re-ranker for the final order. Section 8 draws this wiring; the Extensions section lets you tune the semantic-vs-TF-IDF blend yourself (α=0.5 already pulls Lila into the top 5 for *"blue summer dress"*).

## 8 · Sarah's production wiring diagram

Design for NorthStar's search:

> **What's real vs. illustrative here.** This notebook actually implements the **score-blend re-ranker** — the α-weighted combination of semantic + TF-IDF scores in **E1** (`hybrid_search`). The **popularity** and **recency** signals in the diagram below are *illustrative*: in production you'd add them from clickstream and catalogue-date data, which we don't have in this 76-product demo.

The hybrid design real search systems actually ship: run TF-IDF **and** semantic retrieval in parallel, union the candidates, filter, re-rank, return.

```
                    +-----------------------------+
                    |   Customer types a query    |
                    +-------------+---------------+
                                  |
                  +---------------+---------------+
                  |                               |
        +---------v---------+         +-----------v----------+
        |  TF-IDF retrieval |         | Semantic retrieval   |
        | top-50 by TF-IDF  |         |  top-50 by cosine    |
        +---------+---------+         +-----------+----------+
                  |                               |
                  +---------------+---------------+
                                  |
                    +-------------v-------------+
                    | Union -> 80-100 candidates|
                    |  (counts illustrative --  |
                    |   this demo has 76)       |
                    +-------------+-------------+
                                  |
                    +-------------v-------------+
                    | Structured filters apply: |
                    | category / colour / price |
                    +-------------+-------------+
                                  |
                    +-------------v-------------+
                    | Re-rank by score blend +  |
                    | popularity + recency      |
                    +-------------+-------------+
                                  |
                    +-------------v-------------+
                    |   Top-20 to the customer  |
                    +---------------------------+
```

**What each stage does, and why it's there:**

1. **Customer types a query** — free-text, anything from `"P0047"` to `"something cosy to wear hiking"`. The system can't assume which kind it is, so it runs *both* retrievers every time.
2. **TF-IDF retrieval (top-50)** — the exact-token specialist. Catches SKUs, brand names, and precise product titles that embeddings miss (the `P0047` case from §7). Cheap to run.
3. **Semantic retrieval (top-50)** — the meaning specialist. Catches synonyms and paraphrases that TF-IDF misses (`jumper`→`Cable Jumper`, `cosy`→`warm`). Runs in parallel with TF-IDF, not after it.
4. **Union → ~80–100 candidates** — pool both shortlists into one candidate set. Recall-maximising: a product needs to be found by *either* engine to survive, so neither engine's blind spot can drop a good result. (Counts are illustrative — this 76-product demo can't actually reach 100.)
5. **Structured filters (category / colour / price)** — apply hard business constraints the rankers can't express. `price < £100` or `in stock` is a yes/no fact, not a similarity score — so it's enforced here, *after* retrieval, by removing rows. This is the `search_with_filters` logic from §4.
6. **Re-rank by score blend + popularity + recency** — produce the final order. Blend the semantic and TF-IDF scores (the `hybrid_search` α-weighting in **E1**), then nudge with business signals: popular and recently-added products rise. *Score blend is implemented in this notebook; popularity and recency are illustrative — they'd come from clickstream and catalogue-date data we don't have in this demo.*
7. **Top-20 to the customer** — return the final, filtered, re-ranked page.

The takeaway: production search isn't "semantic **or** keyword" — it's **both, plus filters, plus a re-ranker**. Each layer covers a different failure mode of the others.

**Practical considerations Sarah flagged for production:**

- **Embedding latency.** Encoding the query is ~5 ms on CPU. Cosine over 50K products is < 5 ms with a NumPy matmul. Total search latency < 50 ms — fine.
- **Re-embedding cadence.** When merchandisers update a product description, that product's embedding goes stale. Re-embed the affected rows nightly.
- **Cold-start.** When NEW products arrive, embed them within minutes (a separate small batch job) so they appear in search quickly.
- **Vector database.** At < 100K products, NumPy is fine. At 1M+, switch to FAISS / Pinecone / Weaviate for sub-millisecond approximate-nearest-neighbour.
- **Monitoring.** Log query → top-result CTRs. If a query's top results have zero clicks → bad ranking → re-rank weights need tuning.

## 9 · Recap

In this notebook we:

1. Built a `SemanticSearch` class with a clean three-method API
2. Added structured filters on top of semantic ranking — *exactly* what production search needs
3. Compared head-to-head against TF-IDF and saw both have niches (synonyms vs exact tokens)
4. Drew the production diagram: hybrid retrieval + filters + re-ranker

You now have a working semantic search engine. The next lesson opens the black box: what's INSIDE the transformer that produces these embeddings?

---

## 🟡 Extension — self-study after class

*Skipping this section will not affect your understanding of later lessons. Come back to it when you have time and want to go deeper.*

## E1 · A simple hybrid score

Combine semantic and TF-IDF scores linearly. Tune the weight and see what works best.

In [10]:
def hybrid_search(query, top=5, alpha=0.5):
    """Hybrid = alpha * semantic + (1-alpha) * tfidf"""
    q_sem = model.encode([query])
    sem_sims = cosine_similarity(q_sem, catalogue_emb)[0]

    q_tf = tfidf.vec.transform([query])
    tf_sims = cosine_similarity(q_tf, tfidf.matrix)[0]

    # Both are roughly in [0, 1] — but absolute scales differ.
    # Min-max normalise within this query's candidates so neither dominates.
    def mm(x):
        lo, hi = x.min(), x.max()
        return (x - lo) / (hi - lo + 1e-9)

    blend = alpha * mm(sem_sims) + (1 - alpha) * mm(tf_sims)  # alpha weights semantic vs TF-IDF
    order = np.argsort(-blend)[:top]
    return df.iloc[order].assign(
        score=blend[order],
        sem_score=sem_sims[order],
        tf_score=tf_sims[order]
    ).reset_index(drop=True)

# Sweep alpha from 0 (pure TF-IDF) to 1 (pure semantic); sweet spot sits around 0.6-0.8
for alpha in [0.0, 0.5, 1.0]:
    print(f"\nalpha = {alpha:.1f}  (0=pure TF-IDF, 1=pure semantic)")
    print(f"Query: 'blue summer dress'")
    print(hybrid_search('blue summer dress', alpha=alpha)[['name','score','sem_score','tf_score']].to_string(index=False))


alpha = 0.0  (0=pure TF-IDF, 1=pure semantic)
Query: 'blue summer dress'
                   name    score  sem_score  tf_score
Holly Knit Jumper Dress 1.000000   0.466092  0.147678
      Marina Wrap Dress 0.980256   0.403601  0.144762
      Frost Linen Shirt 0.845372   0.547784  0.124843
   Cobalt V-Neck Jumper 0.827169   0.401502  0.122155
Sand Espadrille Sandals 0.804894   0.265828  0.118865

alpha = 0.5  (0=pure TF-IDF, 1=pure semantic)
Query: 'blue summer dress'
                   name    score  sem_score  tf_score
      Frost Linen Shirt 0.922686   0.547784  0.124843
Holly Knit Jumper Dress 0.916393   0.466092  0.147678
      Marina Wrap Dress 0.842566   0.403601  0.144762
   Lila Floral Sundress 0.765718   0.467017  0.102895
   Cobalt V-Neck Jumper 0.763874   0.401502  0.122155

alpha = 1.0  (0=pure TF-IDF, 1=pure semantic)
Query: 'blue summer dress'
                   name    score  sem_score  tf_score
      Frost Linen Shirt 1.000000   0.547784  0.124843
   Marigold Shift Dres

Run the cell with α=0.0, α=0.5, α=1.0 to see the spectrum. The sweet spot for most production systems sits around 0.6-0.8 — slightly biased toward semantic but TF-IDF helping with exact matches.

## E2 · Find duplicates within the catalogue

Catalogue dedup is a real merchandising problem. Embed every product and look for pairs with very high cosine similarity — those are candidates for being near-duplicates.

In [11]:
# Duplicate detection: compare every product against every other and flag the closest pairs.
# In a real catalogue, pairs above ~0.95 are likely near-duplicates worth merging.
all_sims = cosine_similarity(catalogue_emb)  # all-pairs cosine
np.fill_diagonal(all_sims, -1)  # exclude self-similarity (every item matches itself perfectly)

# Find the top-3 most similar product pairs
flat_idx = np.argsort(-all_sims.ravel())[:6]
seen = set()
print("Top product-pair similarities (potential duplicates):")
for fi in flat_idx:
    i, j = divmod(fi, len(df))
    if (j, i) in seen: continue  # each pair shows up twice (i,j) and (j,i) — keep one
    seen.add((i, j))
    print(f"  {all_sims[i,j]:.3f}  {df.iloc[i]['name']:35s} <-> {df.iloc[j]['name']}")

Top product-pair similarities (potential duplicates):
  0.695  Aspen Snow Boots                    <-> Boulder Hiking Boots
  0.609  Iron Combat Boots                   <-> Loam Ankle Boots
  0.604  Cycle Bib Shorts                    <-> Drift Running Shorts


None of these will actually be true duplicates in our catalogue, but you can see how the technique would flag them. In a real product taxonomy, near-duplicates often share a backbone description (manufacturer copy) — their cosine similarity sits above 0.95. Pairs over that threshold get sent to merchandisers to merge or differentiate.